# Notebook 1: Cut the Kade pages into images for training

This notebook gets the Kade page images and turns them into the pictures the model learns from,
each paired with the matching part of the correct transcription, ready for Notebook 2.

You choose how the pages are cut, in Step 4:

- **`"lines"`**: one image per line of writing. The Kurrent adapter we build on was trained on
  single lines, so this is closest to what it knows.
- **`"groups"`**: a few lines per image (3 by default). Fewer edges and more context.
- **`"pages"`**: the whole page, uncut.

For lines and groups, **kraken** first finds where each line is. Each image is then cut as a strip
that follows the slope of the writing, reaching up toward the line above and down toward the line
below, so the dots on i and j, the umlauts, the u-marks, the tall loops of l, h and ß and the tails
of g, f and h all stay in. Where the neighbouring lines' writing would show, the image is filled
with the colour of the paper, so each image shows only its own line(s).

You run kraken once for the whole project (about 3 hours). After that you can change how the
pages are cut in a few minutes, without running kraken again.

Run the cells from top to bottom with **Shift + Enter** (the same on Mac and Windows).
Kernel: **conda_pytorch_p310** (or whichever `conda_pytorch` kernel your instance lists).

## Step 1. Set up the folders

This cell decides where everything goes. You don't need to change anything in it.

In [ ]:
import importlib.util
import os
import subprocess
import sys

# Which kernel is this? It should be conda_pytorch_p310 (or another conda_pytorch kernel).
kernel = os.path.basename(os.path.dirname(os.path.dirname(sys.executable)))
print("Kernel environment:", kernel)
# The notebooks need Pillow (images) and pandas (tables) in this kernel. If it
# doesn't have them, install them here.
missing = [pkg for module, pkg in (("PIL", "pillow"), ("pandas", "pandas"))
           if importlib.util.find_spec(module) is None]
if missing:
    print("Installing", ", ".join(missing), "into this kernel ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# Everything this project downloads or creates lives here. It is on the
# notebook instance's storage volume, so it survives Stop and Start.
WORK = "/home/ec2-user/SageMaker/badger-finetune"
DATA = os.path.join(WORK, "data")            # page images, labels, line images
RUNS = os.path.join(WORK, "runs")            # one folder per training run
SCRIPTS = os.path.abspath("scripts")         # the scripts folder next to this notebook
for folder in (WORK, DATA, RUNS):
    os.makedirs(folder, exist_ok=True)
sys.path.insert(0, SCRIPTS)

# Keep the 9 GB model download on the big volume, not the small system disk.
os.environ["HF_HOME"] = os.path.join(WORK, "hf_cache")

SHARED_BUCKET = "ml-marathon-shared"   # the class bucket with the Badger Scribe data
SHARED_PREFIX = "badger-scribe"
print("Working folder:", WORK)

## Step 2. Install kraken (only the first time)

kraken needs a newer version of PyTorch than the one this notebook's kernel uses for training. If
we installed it here, it would replace that version and break Notebook 2. So kraken gets its own,
separate Python environment inside the working folder. We never switch to it: when it's time to
cut pages, Notebook 1 runs the kraken script with that environment's Python.

The environment lives on the instance's storage volume, so this cell only does real work the
first time (about 3 minutes). After that it says the environment is ready and finishes in seconds.

In [ ]:
import subprocess

KRAKEN_ENV = os.path.join(WORK, "kraken-env")
KRAKEN_PYTHON = os.path.join(KRAKEN_ENV, "bin", "python")

# uv downloads Python and packages. Point its downloads at the working folder,
# so they survive a Stop too.
os.environ["UV_PYTHON_INSTALL_DIR"] = os.path.join(WORK, "uv-python")
os.environ["UV_CACHE_DIR"] = os.path.join(WORK, "uv-cache")

def kraken_works():
    test = "import kraken, importlib.metadata as m; print('kraken', m.version('kraken'))"
    return os.path.exists(KRAKEN_PYTHON) and subprocess.run([KRAKEN_PYTHON, "-c", test]).returncode == 0

if kraken_works():
    print("The kraken environment is ready.")
else:
    print("Setting up the kraken environment. This takes about 3 minutes ...")
    !{sys.executable} -m pip install -q uv
    !{sys.executable} -m uv venv -q --python 3.12 {KRAKEN_ENV}
    !{sys.executable} -m uv pip install -q --python {KRAKEN_PYTHON} --torch-backend cpu kraken
    if not kraken_works():
        raise SystemExit("kraken didn't install. Look at the messages above, or see Troubleshooting in README.md.")
    print("The kraken environment is ready.")

## Step 3. Download the page images and labels

This copies the Kade page images and the two lists of pages from S3 to the instance:
`train.csv` (the 295 pages with correct transcriptions) and `test.csv` (the 141 pages Kaggle
scores us on).

**Tell it where the data is** with the two lines at the top of the cell. The defaults are the
class bucket. If the data is somewhere else, for example in your own bucket (README.md, "Where
the data comes from"), change them. The folder must hold `train.csv`, `test.csv` and an
`images/` folder.

If the files aren't where you said, the cell **searches the bucket** and prints where it found
`train.csv`, so you know what to type. Images already downloaded are skipped, so it's safe to
run again.

In [ ]:
import glob

import boto3
import pandas as pd
from botocore.exceptions import ClientError

DATA_BUCKET = SHARED_BUCKET   # the bucket the data is in, e.g. "your-bucket-name"
DATA_PREFIX = SHARED_PREFIX   # the folder inside it that holds train.csv, test.csv and images/

s3 = boto3.client("s3")
images_dir = os.path.join(DATA, "images")
os.makedirs(images_dir, exist_ok=True)

def find_in_bucket(ending, most=10):
    # Where in the bucket are files whose names end like this?
    found = []
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=DATA_BUCKET):
        found += [o["Key"] for o in page.get("Contents", []) if o["Key"].endswith(ending)]
        if len(found) >= most:
            break
    return found[:most]

def explain(error, key):
    code = error.response.get("Error", {}).get("Code", "")
    if code in ("404", "NoSuchKey", "NotFound"):
        try:
            hits = find_in_bucket("train.csv")
        except ClientError:
            hits = None
        if hits:
            folders = sorted({h.rsplit("/", 1)[0] if "/" in h else "" for h in hits})
            return (f"There is no {key} in the bucket {DATA_BUCKET}. train.csv is in: "
                    + ", ".join(repr(f) for f in folders) + ". Set DATA_PREFIX to one of these and run again.")
        if hits == []:
            return (f"The bucket {DATA_BUCKET} has no train.csv anywhere, so the data hasn't been put in it. "
                    "Upload it to your own bucket (README.md, 'Where the data comes from') and set DATA_BUCKET.")
        return f"There is no {key} in the bucket {DATA_BUCKET}, and this notebook isn't allowed to look inside it."
    if code in ("403", "AccessDenied"):
        return (f"This notebook isn't allowed to read {DATA_BUCKET}. Ask the class organisers to give your "
                "notebook's role read access, or put the data in your own bucket (README.md).")
    if code == "NoSuchBucket":
        return f"There is no bucket called {DATA_BUCKET}. Check the name, in the S3 console."
    return f"S3 said: {error}"

for name in ("train.csv", "test.csv"):
    key = f"{DATA_PREFIX}/{name}" if DATA_PREFIX else name
    try:
        s3.download_file(DATA_BUCKET, key, os.path.join(DATA, name))
    except ClientError as e:
        raise SystemExit(explain(e, key))

train = pd.read_csv(os.path.join(DATA, "train.csv"), keep_default_na=False)
test = pd.read_csv(os.path.join(DATA, "test.csv"), keep_default_na=False)
# Only the Kade pages: a full Kaggle download also lists the survey and account-book pages.
wanted = [p for p in pd.concat([train, test])["page_id"] if p.startswith("kade_")]
prefix = f"{DATA_PREFIX}/images/" if DATA_PREFIX else "images/"
new, missing = 0, []
for page_id in wanted:
    path = os.path.join(images_dir, page_id + ".jpg")
    if os.path.exists(path):
        continue
    try:
        s3.download_file(DATA_BUCKET, prefix + page_id + ".jpg", path)
        new += 1
    except ClientError:
        missing.append(page_id)
n_train, n_test = (int(df["page_id"].str.startswith("kade_").sum()) for df in (train, test))
print(f"{n_train} Kade training pages, {n_test} Kade test pages; {new} images downloaded now, "
      f"{len(glob.glob(os.path.join(images_dir, '*.jpg')))} in {images_dir}")
if missing:
    hits = find_in_bucket(missing[0] + ".jpg", most=1)
    where = f" It is at {hits[0]!r}, so the images folder is somewhere else." if hits else ""
    print(f"WARNING: {len(missing)} images aren't at {prefix}<page>.jpg, e.g. {missing[0]}.{where}")

## Step 4. Choose how to cut the pages

| `CROP` | Each image holds | Good | Less good |
|---|---|---|---|
| `"lines"` | one line | What the Kurrent adapter was trained on; many small examples to learn from | Only pages where kraken finds exactly as many lines as the transcription has can be used for training |
| `"groups"` | `GROUP` lines (3 by default) | Fewer cut edges, more context; a middle way | Same rule about matching line counts |
| `"pages"` | the whole page | Nothing is cut at all, every page can be used, no kraken needed | Far from how the adapter was trained, needs much more GPU memory and time, and the page is shrunk to fit |

`ABOVE` and `BELOW` set how far each strip reaches toward the neighbouring lines, as a share of
the space between two lines (0 = not at all, 1 = all the way). The defaults keep dots, loops and
tails without taking in the next line's writing. If Step 7 shows the tops of tall letters cut off,
raise `ABOVE` a little (say to 0.8); tails cut off at the bottom, raise `BELOW` (say to 0.6). If
you see a lot of the neighbouring lines, lower them.

You can come back and change these at any time: run Steps 4, 6, 7 and 8 again (not Step 5).
Runs you trained on one kind of image can only read that same kind, so Notebook 3 checks that
they match.

In [ ]:
CROP = "lines"    # "lines", "groups" or "pages"
GROUP = 3         # lines per image, for "groups"
ABOVE = 0.7       # how far each image reaches up toward the line above (0 to 1)
BELOW = 0.5       # how far it reaches down toward the line below (0 to 1)

## Step 5. Find the lines with kraken

kraken looks at each page and finds where every line of writing is. It saves those positions in
`data/kade_segments/`, one small file per page; nothing is cut yet. This is the slow part, so it's
done only once.

**Start small.** `LIMIT = 5` does just five pages, so you can check the result in Step 7 first.
Then set `LIMIT = None` and run this cell again for all of them. Pages already done are skipped,
so nothing is repeated, and if it stops halfway you can simply run it again.

It works on several pages at once, one per processor. On a 4-processor instance (like
`ml.g4dn.xlarge`), expect about **3 hours** for all 436 pages. The last number on each line is the
time left. You can close the browser meanwhile; it keeps going while the instance runs.

With `CROP = "pages"` kraken isn't needed, and this cell does nothing.

In [ ]:
LIMIT = 5   # then None for every page

segments_dir = os.path.join(DATA, "kade_segments")
if CROP == "pages":
    print("CROP is 'pages': no need to find lines. Go on to Step 6.")
else:
    limit = f"--limit {LIMIT}" if LIMIT else ""
    !{KRAKEN_PYTHON} {SCRIPTS}/segment_lines.py --images {images_dir} --out {segments_dir} {limit}

## Step 6. Cut the images

This cuts the pages the way you chose in Step 4, into `data/kade_lines/<page name>/000.png,
001.png, ...` (top to bottom). It replaces whatever was there before, so the folder only ever
holds one kind of image. It takes a few minutes for all pages.

For lines and groups it also puts back together lines that kraken split up. Where a line has a gap
in it (a detached capital, a wide space), kraken sometimes returns two or three short pieces side
by side, and those are joined into one line. For a training page, it tries a few ways of joining
and keeps the one whose number of lines matches the transcription. Dev and test pages are always
joined the same default way, without looking at their transcription, so their scores stay honest.

In [ ]:
lines_dir = os.path.join(DATA, "kade_lines")
python = sys.executable
!{python} {SCRIPTS}/make_crops.py --images {images_dir} --segments {segments_dir} --out {lines_dir} --mode {CROP} --group {GROUP} --above {ABOVE} --below {BELOW} --labels {DATA}/train.csv

## Step 7. Check the result by eye

This shows a page with a box around each image that was cut from it (the boxes overlap at the
edges, as they should, since each strip reaches toward its neighbours), then the first few images
themselves. Check that:

- every dot, umlaut, loop and tail is inside its image, not cut off at the edge;
- each image holds the right number of lines: one for `"lines"`, `GROUP` for `"groups"`.

A few mistakes are fine; Step 8 leaves pages whose line counts don't match out of training. If
letters are cut off at the top, raise `ABOVE` in Step 4, then run Steps 6 and 7 again; at the
bottom, raise `BELOW`. Try another page by typing its name.

In [ ]:
import pandas as pd
from IPython.display import display
from PIL import Image, ImageDraw

crops = pd.read_csv(os.path.join(lines_dir, "crops.csv"), keep_default_na=False)
page = crops.page_id.iloc[0]   # or type a page name, e.g. "kade_112_p004"

boxes = crops[crops.page_id == page]
img = Image.open(os.path.join(images_dir, page + ".jpg")).convert("RGB")
draw = ImageDraw.Draw(img)
for i, b in enumerate(boxes.itertuples()):
    draw.rectangle((b.left, b.top, b.right - 1, b.bottom - 1), outline=("red", "blue")[i % 2], width=3)
img.thumbnail((700, 1000))
print(f"{page}: {len(boxes)} images")
display(img)
for b in boxes.head(6).itertuples():
    print(b.crop)
    display(Image.open(os.path.join(lines_dir, b.crop)))

## Step 8. Pair the images with the transcriptions

For training we need to know which part of the transcription belongs to which image:

- **pages**: each page image simply gets its whole transcription, so every page is used.
- **lines**: when kraken found **exactly as many lines** on a page as its transcription has, image 1
  gets line 1, image 2 gets line 2, and so on. Pages where the numbers differ are left out of
  training, since we can't be sure which line is which.
- **groups**: the same rule; each image gets the lines it holds (for example lines 4 to 6).

Two things are taken out of the transcriptions first: blank lines, and the `Postmark ...` line some
postcards have (the postmark is on the back of the card, so it isn't in the image).

The 11 pages of letter `kade_112` are the team's **dev set**: never trained on, only used to score
the model.

**Read the first line it prints.** For lines or groups, if well under 70% of training pages are
usable, look at some of the pages listed below it in Step 7 before training.

In [ ]:
import prepare_training_data

counts = prepare_training_data.build(DATA, os.path.join(DATA, "train.csv"))
bad = counts[(~counts["match"]) & (counts["kraken_lines"] > 0)]
if len(bad):
    print("\nSome pages whose line counts don't match (kraken vs. transcription):")
    print(bad.head(10)[["page_id", "kraken_lines", "label_lines"]].to_string(index=False))
else:
    print("\nEvery page that was cut matches its transcription.")

## Step 9. Save to S3 for the team (optional)

The images are on this instance only. Copy them to your own bucket so teammates (or you, on a new
instance) can skip this notebook. Fill in your bucket's name first.

In [ ]:
MY_BUCKET = "your-bucket-name"   # <-- your own bucket

!aws s3 sync {DATA} s3://{MY_BUCKET}/badger-finetune/data --exclude "images/*" --quiet
print(f"Saved to s3://{MY_BUCKET}/badger-finetune/data")

**Done.** Go on to **Notebook 2** (`2_finetune.ipynb`).

To use someone else's results instead of running this notebook, run Step 1, then in a cell:
`!aws s3 sync s3://<their bucket>/badger-finetune/data {DATA}`